# Experimentos: optimización de hiperparámetros con Optuna

Objetivo: usar Optuna para buscar los mejores hiperparámetros de XGBoost,
rastrear cada trial como un run anidado en MLflow, entrenar el modelo final
con los mejores parámetros y registrarlo en MLflow Model Registry.

In [1]:
from __future__ import annotations

import warnings

import mlflow
import pandas as pd

from healthcare_fraud.data import clean_dataframe, load_dataset, validate_dataframe
from healthcare_fraud.features import (
    FEATURE_COLS,
    build_features,
    prepare_train_val,
    split_providers,
)
from healthcare_fraud.models import (
    optimize_hyperparameters,
    register_model,
    setup_mlflow,
    train_model,
    transition_model_stage,
)

warnings.filterwarnings("ignore")
print("Imports OK")

D:\Diego\projects\mlops_udem\fraude-mlops\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports OK


## 1. Preparación de datos

In [2]:
raw_tables = load_dataset()
clean_tables = {
    name: clean_dataframe(validate_dataframe(df, name), name) for name, df in raw_tables.items()
}
print("Tablas limpias:", {k: v.shape for k, v in clean_tables.items()})

Unrecognized CSV filename: Test_Beneficiarydata-1542969243754.csv — skipped
Unrecognized CSV filename: Test_Inpatientdata-1542969243754.csv — skipped
Unrecognized CSV filename: Test_Outpatientdata-1542969243754.csv — skipped
[outpatient] High null percentage: 57.90%


Tablas limpias: {'labels_test': (1353, 1), 'labels_train': (5410, 2), 'beneficiary': (138556, 24), 'inpatient': (40474, 23), 'outpatient': (517737, 14)}


In [3]:
feature_df = build_features(clean_tables)
train_df, val_df = split_providers(feature_df)

X_train, X_val, y_train, y_val, preprocessor = prepare_train_val(train_df, val_df)
X_train_raw = train_df[FEATURE_COLS].values
X_val_raw = val_df[FEATURE_COLS].values

n_fraud = int(y_train.sum())
n_total = len(y_train)
print(f"Train: {n_total} providers | Fraude: {n_fraud} ({n_fraud / n_total:.1%})")

Train: 4328 providers | Fraude: 405 (9.4%)


## 2. Optimización de hiperparámetros con Optuna

In [4]:
setup_mlflow()

with mlflow.start_run(run_name="optuna_experiment") as parent_run:
    mlflow.set_tag("notebook", "03_experiments")
    best_params = optimize_hyperparameters(X_train, y_train, X_val, y_val)

print("Mejores hiperparámetros:")
pd.Series(best_params)

2026/04/27 22:10:24 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/04/27 22:10:24 INFO mlflow.store.db.utils: Updating database tables
2026/04/27 22:10:28 INFO mlflow.tracking.fluent: Experiment with name 'healthcare-fraud-detection' does not exist. Creating a new experiment.


Mejores hiperparámetros:


n_estimators        436.000000
max_depth             3.000000
learning_rate         0.019454
subsample             0.747395
colsample_bytree      0.796916
min_child_weight      8.000000
reg_alpha             2.324142
reg_lambda            1.923350
dtype: float64

## 3. Entrenamiento del modelo final

In [5]:
with mlflow.start_run(run_name="final_experiment") as parent_run:
    mlflow.set_tag("notebook", "03_experiments")
    run_id, metrics = train_model(X_train_raw, y_train, X_val_raw, y_val, preprocessor, best_params)

print(f"Run ID: {run_id}")
pd.DataFrame([metrics], index=["optimizado"]).round(4)

2026/04/27 22:10:51 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/27 22:10:51 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/27 22:11:09 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/04/27 22:11:10 INFO mlflow.models.model: Found the following environment variables used during model inference: [GEMINI_API_KEY, KAGGLE_API_TOKEN]. Please check if you need to set them when deploying the model. To disable this message, set environment variable `MLFLOW_RECORD_ENV_VARS_IN_MODEL_LOGGING` to `fal

Run ID: de1f1c262af34c859429c82a7eb5c3e9


,recall,precision,f1,roc_auc,avg_precision
optimizado,0.9109,0.436,0.5897,0.9643,0.7549


## 4. Registro en MLflow Model Registry

In [6]:
MODEL_NAME = "healthcare-fraud-detector"

mv = register_model(run_id, MODEL_NAME)
print(f"Modelo registrado: {mv.name} v{mv.version}")

Successfully registered model 'healthcare-fraud-detector'.
2026/04/27 22:11:10 WARNING mlflow.tracking._model_registry.fluent: Run with id de1f1c262af34c859429c82a7eb5c3e9 has no artifacts at artifact path 'model', registering model based on models:/m-edc51aece0704302b1ba38fa96658d34 instead


Modelo registrado: healthcare-fraud-detector v1


Created version '1' of model 'healthcare-fraud-detector'.


In [7]:
mv_staging = transition_model_stage(MODEL_NAME, mv.version, "Staging")
print(f"Stage: {mv_staging.current_stage}")

Stage: Staging


## Conclusiones

- Los runs de Optuna quedan registrados como runs anidados en MLflow UI.
- Para ver los resultados: `uv run mlflow ui --backend-store-uri sqlite:///mlflow.db`
- El modelo optimizado está en MLflow Registry en stage **Staging**.
- El siguiente paso (Fase 03) es orquestar este flujo completo con Prefect
  y exponer el modelo vía FastAPI en la Fase 04.